# ⚡ Etapa 2: Optimización y Conversión a ONNX

Este notebook actúa como el motor de transformación del pipeline. Su objetivo es convertir el modelo entrenado en **PyTorch** a un formato **ONNX (Open Neural Network Exchange)**, optimizándolo para su despliegue en entornos de producción de alta demanda.

---

### 📦 1. Preparación de la Infraestructura
Al igual que en las etapas anteriores, aseguramos la autonomía del nodo:
* **Instalación de Motores de Inferencia:** Instalamos `optimum` y `onnxruntime`, herramientas diseñadas para ejecutar grafos de redes neuronales con máxima eficiencia en CPU y GPU.
* **Workspace Limpio:** Establecemos el directorio de trabajo en `/opt/app-root/src` para garantizar la compatibilidad con el sistema de archivos de **OpenShift AI**.

### 🔓 2. Desempaquetado de Artefactos (Data Handoff)
El pipeline de **Elyra** nos entrega el archivo `modelo_pytorch.tar.gz` generado en la etapa anterior. 
* En este paso, extraemos los pesos originales del modelo para que el convertidor pueda leer el grafo de la red neuronal y sus pesos binarios.

### 🔄 3. Proceso de Exportación a ONNX
Utilizamos **Hugging Face Optimum** para realizar la traducción del modelo:
* **Exportación Nativa:** El parámetro `export=True` rastrea las operaciones de **BETO** y las convierte en operadores ONNX universales.
* **Consistencia del Tokenizer:** Guardamos el tokenizador junto al modelo optimizado para asegurar que el pre-procesamiento del texto no varíe entre el entrenamiento y la inferencia real.

### 📦 4. Generación del Artefacto de Producción
Finalmente, el modelo optimizado se empaqueta en `modelo_onnx.tar.gz`. 
* Este archivo comprimido es mucho más eficiente para ser transportado y es el formato final que recibirá el servidor de modelos (**ModelMesh**) o la etapa de subida a **MinIO**.

---
> **Estado:** Al concluir este proceso, el modelo ha dejado de ser un archivo de investigación pesado para convertirse en un motor de inferencia ágil y listo para escalar en el clúster.

In [ ]:
import os, tarfile, subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "optimum[onnxruntime]", "transformers", "torch"])

from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

os.chdir("/opt/app-root/src")

with tarfile.open("modelo_pytorch.tar.gz", "r:gz") as tar:
    tar.extractall()

print("🚀 Convirtiendo a ONNX...")
model_in = "modelo_t-moviles_pytorch"
model_out = "modelo_t-moviles_onnx"

tokenizer = AutoTokenizer.from_pretrained(model_in)
model = ORTModelForSequenceClassification.from_pretrained(model_in, export=True)

model.save_pretrained(model_out)
tokenizer.save_pretrained(model_out)

# Empaquetar ONNX
with tarfile.open("modelo_onnx.tar.gz", "w:gz") as tar:
    tar.add(model_out, arcname=os.path.basename(model_out))
print("✅ ONNX empaquetado.")